# TP 1 / 10 — Exploration des données freMPL

**M2 Actuariat — Abidjan**
**Cours : Data Science / Machine Learning en assurance + Tarification & Crédibilité**

---

Ce TP est le premier d'une série de 10 notebooks qui couvrent l'intégralité du cycle de vie d'un modèle de **tarification IARD** sur le jeu de données français `freMPL` (assurance automobile, source CASdatasets).

## Objectifs du TP 1
- Comprendre le contexte métier du dataset.
- Faire une **analyse exploratoire (EDA) complète** : distribution, valeurs manquantes, corrélations.
- Repérer les pièges spécifiques aux données d'assurance (déséquilibre, fuites de variables, modalités rares).
- Préparer le terrain pour le TP 2 (preprocessing).

## Plan de la série (10 TP)
1. **EDA** (ce notebook)
2. Preprocessing & feature engineering
3. GLM logistique (cours de tarification)
4. Évaluation actuarielle (lift, Gini)
5. GLM amélioré (interactions, régularisation, seuil)
6. Crédibilité & stabilité (bootstrap)
7. Modèles ML avancés (arbre, RF, LightGBM)
8. Interprétabilité (SHAP vs coefficients)
9. Équité & calibration par groupe
10. Cycle de vie ML & déploiement

**Durée estimée pour ce TP : ~45 min.**

---


## 0. Préambule — imports

Nous restons délibérément sur des outils standards. **Pas de `Pipeline` scikit-learn** dans toute la série : chaque étape est explicite pour bien comprendre ce qui se passe.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)


## 1. Contexte métier

Le dataset `freMPL` est un échantillon de **contrats d'assurance automobile français**, largement utilisé pour enseigner la tarification non-vie. Chaque ligne représente **un contrat** observé sur une période. Les variables couvrent :

- **L'assuré** : âge (`DrivAge`), ancienneté du permis (`LicAge`), sexe (`Gender`), statut marital (`MariStat`), catégorie socio-professionnelle (`SocioCateg`).
- **Le véhicule** : âge (`VehAge`), carrosserie (`VehBody`), prix (`VehPrice`), motorisation (`VehEngine`, `VehEnergy`), vitesse max (`VehMaxSpeed`), classe (`VehClass`), garage.
- **L'usage et l'historique** : usage du véhicule (`VehUsage`), limite kilométrique (`HasKmLimit`), **coefficient bonus-malus** (`BonusMalus`), variable de risque maison (`RiskVar`).
- **La cible** : `y` ∈ {`GOOD`, `BAD`}. Un contrat est dit `BAD` s'il a généré au moins un sinistre coûteux sur la période d'observation, `GOOD` sinon.

> Dans la suite, nous traiterons donc un **problème de classification binaire** : prédire la probabilité qu'un nouveau contrat soit `BAD`. C'est l'équivalent simplifié d'un modèle de **fréquence** en tarification IARD.


## 2. Chargement et premier aperçu


In [ ]:
df = pd.read_csv("../freMPL.csv")
print("Dimensions :", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


### Question 1
- Combien de contrats et combien de variables ?
- Combien de variables numériques vs catégorielles ?
- Y a-t-il des valeurs manquantes visibles ? Sur quelle(s) colonne(s) ?

*Votre réponse :*


## 3. Analyse de la variable cible `y`

C'est **toujours** la première chose à regarder : à quoi ressemble ce qu'on cherche à prédire ?


In [ ]:
# Recodage en binaire : BAD = 1 (sinistre), GOOD = 0 (pas de sinistre)
df["target"] = (df["y"] == "BAD").astype(int)

counts = df["target"].value_counts()
freqs  = df["target"].value_counts(normalize=True)

print("Effectifs :")
print(counts)
print("\nFréquences :")
print(freqs.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x="target", data=df, palette="viridis", ax=ax)
ax.set_xticklabels(["GOOD (0)", "BAD (1)"])
ax.set_title("Distribution de la cible — déséquilibre de classes")
ax.set_ylabel("Nombre de contrats")
plt.show()


### Question 2
- Quel est le **taux de sinistralité** dans ce portefeuille ?
- Si on prédit naïvement « tous les contrats sont GOOD », quelle accuracy obtient-on ? Pourquoi cette métrique est-elle **trompeuse** ici ?
- Citez **deux métriques** plus adaptées à un problème déséquilibré.

*Votre réponse :*


## 4. Variables numériques — distributions et lien avec la cible

Variables strictement numériques : `LicAge` (ancienneté permis en mois), `DrivAge` (âge), `BonusMalus`, `RiskVar`, `HasKmLimit`.


In [ ]:
num_vars = ["LicAge", "DrivAge", "BonusMalus", "RiskVar", "HasKmLimit"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, v in enumerate(num_vars):
    sns.histplot(data=df, x=v, hue="target", bins=30, kde=False,
                 stat="density", common_norm=False, ax=axes[i], palette="viridis")
    axes[i].set_title(f"Distribution de {v} selon la cible")
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots conditionnels : plus lisibles pour comparer les distributions
fig, axes = plt.subplots(1, len(num_vars), figsize=(20, 4))
for i, v in enumerate(num_vars):
    sns.boxplot(data=df, x="target", y=v, palette="viridis", ax=axes[i])
    axes[i].set_title(v)
    axes[i].set_xticklabels(["GOOD", "BAD"])
plt.tight_layout()
plt.show()


In [ ]:
# Taux de sinistralité par quintile de chaque variable numérique
# (équivalent d'une "courbe de risque" élémentaire en tarification)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, v in enumerate(num_vars):
    tmp = df.copy()
    try:
        tmp["bin"] = pd.qcut(tmp[v], q=5, duplicates="drop")
    except ValueError:
        tmp["bin"] = tmp[v]
    rate = tmp.groupby("bin")["target"].agg(["mean", "count"])
    rate["mean"].plot(kind="bar", ax=axes[i], color="steelblue", edgecolor="k")
    axes[i].set_title(f"Taux de BAD par quintile de {v}")
    axes[i].set_ylabel("P(BAD)")
    axes[i].axhline(df["target"].mean(), color="red", linestyle="--", label="Moyenne globale")
    axes[i].legend()
for j in range(len(num_vars), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


### Question 3
- Pour `DrivAge`, à quelle tranche d'âge le taux de BAD est-il maximal ? Cela correspond-il à l'intuition actuarielle des « jeunes conducteurs » ?
- `BonusMalus` est-il, sans surprise, très lié à `target` ? **Discutez du risque de fuite d'information (data leakage)** : le bonus-malus reflète la sinistralité **passée** ; est-ce un prédicteur légitime pour la sinistralité **future** ?
- `LicAge` et `DrivAge` semblent-ils mesurer la même chose ? Comment le vérifier ?

*Votre réponse :*


## 5. Variables catégorielles — taux de sinistralité par modalité

Pour chaque variable qualitative, on calcule **le taux de BAD par modalité**, et on le compare à la moyenne globale. C'est l'analyse univariée typique d'un actuaire.


In [ ]:
cat_vars = ["Gender", "MariStat", "VehUsage", "VehBody", "VehPrice",
            "VehEngine", "VehEnergy", "VehClass", "Garage"]

base_rate = df["target"].mean()
print(f"Taux moyen de BAD dans le portefeuille : {base_rate:.3%}")

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()
for i, v in enumerate(cat_vars):
    rate = df.groupby(v, dropna=False)["target"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    rate["mean"].plot(kind="bar", ax=axes[i], color="steelblue", edgecolor="k")
    axes[i].axhline(base_rate, color="red", linestyle="--", label=f"Moyenne ({base_rate:.2%})")
    axes[i].set_title(f"Taux de BAD par {v}")
    axes[i].set_ylabel("P(BAD)")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Compter aussi les effectifs par modalité — crucial pour juger de la crédibilité
for v in cat_vars:
    print(f"\n--- {v} ---")
    print(df[v].value_counts(dropna=False).to_string())


### Question 4
- Citez **trois modalités** qui présentent un taux de BAD très supérieur à la moyenne. Sont-elles bien représentées (effectif suffisant) ou s'agit-il d'effets de petits nombres ?
- Pour `VehBody`, la modalité `bus` est-elle « crédible » au sens où vous l'entendrez dans le cours de **crédibilité** ?
- Quelle modalité de `VehPrice` semble la plus risquée ? Vous y attendiez-vous ?

*Votre réponse :*


## 6. Cas particuliers : `VehAge`, `VehMaxSpeed`, `SocioCateg`

Trois variables nécessitent une attention spécifique avant la modélisation.


In [ ]:
# VehAge contient des modalités textuelles : '10+', '8-9' ...
print(df["VehAge"].value_counts(dropna=False))


In [ ]:
# VehMaxSpeed contient des intervalles textuels
print(df["VehMaxSpeed"].value_counts(dropna=False))


In [ ]:
# SocioCateg : beaucoup de modalités, certaines très rares
sc = df["SocioCateg"].value_counts(dropna=False)
print(f"Nombre de modalités distinctes : {sc.shape[0]}")
print(sc.head(15))
print("...")
print(sc.tail(15))


### Question 5
- Pour `VehAge`, comment proposeriez-vous de **convertir `'10+'` et `'8-9'` en numérique** ? Quels sont les choix possibles et quelles hypothèses sous-jacentes ?
- Pour `VehMaxSpeed`, faut-il la traiter comme une variable **ordinale** (encodée par le milieu de l'intervalle) ou **catégorielle** (one-hot) ? Justifiez.
- Pour `SocioCateg`, on a beaucoup de modalités rares. Citez **deux stratégies** pour gérer cela (vous les implémenterez en TP 2).

*Votre réponse :*


## 7. Valeurs manquantes


In [ ]:
na = df.isna().sum()
na = na[na > 0].sort_values(ascending=False)
print("Variables avec des NA :")
print(na)
print(f"\nProportion de NA sur Garage : {df['Garage'].isna().mean():.1%}")


In [ ]:
# Le taux de BAD est-il différent quand Garage est manquant ?
df["Garage_NA"] = df["Garage"].isna()
print(df.groupby("Garage_NA")["target"].agg(["mean", "count"]))


### Question 6
- Quelle variable a beaucoup de valeurs manquantes ?
- Le fait d'avoir une valeur manquante est-il **informatif** sur la cible ? Comparez précisément les taux ci-dessus : l'écart est-il important rapporté à la moyenne globale (~8.7 %) ?
- Trois stratégies sont envisageables : (a) supprimer la variable, (b) supprimer les lignes, (c) créer une modalité `"Inconnu"`. Que choisiriez-vous **ici** et pourquoi ? (Indice : si le signal porté par NA est faible mais non nul, l'option (c) reste défendable, en gardant `Garage` comme variable.)

*Votre réponse :*


## 8. Corrélations entre variables numériques

On regarde les corrélations linéaires (Pearson) — utile pour anticiper la **multicolinéarité** dans le GLM du TP 3.


In [ ]:
corr = df[num_vars + ["target"]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation (Pearson)")
plt.show()


In [ ]:
# V de Cramér pour mesurer l'association entre paires de variables CATÉGORIELLES
from itertools import combinations
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

results = []
for a, b in combinations(cat_vars, 2):
    results.append((a, b, cramers_v(df[a].fillna("NA"), df[b].fillna("NA"))))
vcra = pd.DataFrame(results, columns=["var1", "var2", "V_Cramer"]).sort_values("V_Cramer", ascending=False)
print(vcra.head(15).to_string(index=False))


### Question 7
- Quelles sont les **deux variables numériques les plus corrélées** entre elles ? Quel risque cela pose-t-il pour un GLM ?
- Côté qualitatif, quelles paires ont un V de Cramér élevé (> 0.4) ? Que cela signifie-t-il concrètement (donnez un exemple métier) ?
- Quelle variable numérique a la **plus forte corrélation avec la cible** ? Cela confirme-t-il l'analyse de la question 3 ?

*Votre réponse :*


## 9. Focus actuariel : `BonusMalus`

Le coefficient bonus-malus est par construction une **synthèse de la sinistralité passée** de l'assuré. La théorie (et la pratique tarifaire française) en font un prédicteur majeur du risque futur. **À vérifier sur ce dataset** :

- son lien avec la cible est-il fort sur **toute** la plage, ou concentré sur la **queue** (assurés malussés) ?
- est-ce un prédicteur **légitime** pour de **nouveaux** assurés (sans historique) ?


> Attention : la conclusion à tirer dépend du **profil du portefeuille** observé. Sur `freMPL`, vous allez voir que la majorité des contrats sont à BM = 50 (bonus maximal historique), ce qui aplatit le signal en moyenne et le concentre sur les BM > 100.

In [ ]:
# Distribution de BonusMalus par classe
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=df, x="BonusMalus", hue="target", bins=40,
             stat="density", common_norm=False, ax=axes[0], palette="viridis")
axes[0].set_title("Distribution de BonusMalus par classe")

# Taux de BAD par tranche de BonusMalus
df["bm_bin"] = pd.cut(df["BonusMalus"], bins=[0, 50, 60, 80, 100, 200])
rate = df.groupby("bm_bin", observed=True)["target"].agg(["mean", "count"])
rate["mean"].plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="k")
axes[1].axhline(df["target"].mean(), color="red", linestyle="--", label="Moyenne")
axes[1].set_title("Taux de BAD par tranche de BonusMalus")
axes[1].set_ylabel("P(BAD)")
axes[1].legend()
plt.tight_layout()
plt.show()

print("\nEffectif par tranche :")
print(rate)


### Question 8
- Regardez le tableau des taux de BAD par tranche. **Sur quelle(s) tranche(s)** l'effet est-il visible ? Quel pourcentage du portefeuille est concerné ? Que peut-on dire de la **moyenne** d'effet de `BonusMalus` versus l'**effet aux extrêmes** ?
- Le `BonusMalus` est-il une variable **endogène** ou **exogène** au sens économique ? Justifiez.
- Si vous deviez tarifer un assuré **nouveau** (sans historique en France), quelle valeur de `BonusMalus` lui attribuer ? Quel problème cela pose-t-il pour votre modèle ?
- Proposez deux versions de modèle pour le TP 3 : **(A)** avec `BonusMalus`, **(B)** sans. Que peut-on apprendre de cette comparaison ?

*Votre réponse :*


## 10. Synthèse du TP 1

Dans ce premier TP, vous avez :
1. Découvert le contexte et la structure du dataset `freMPL`.
2. Quantifié le **déséquilibre de la cible** (~8.7 % de BAD).
3. Identifié plusieurs **variables liées au risque** : `DrivAge`, `BonusMalus` (effet concentré sur la queue > 100), certaines `VehBody` et `VehPrice`.
4. Repéré des pièges techniques : `VehAge` textuel, modalités rares de `SocioCateg`, NA massifs sur `Garage` (mais à signal **faible** sur la cible), **corrélations** `LicAge`/`DrivAge`.
5. Discuté un enjeu de fond : **`BonusMalus` est une fuite partielle** vis-à-vis de la sinistralité passée — un atout pour la performance, un piège pour la modélisation des nouveaux assurés.

### Sauvegarde pour les TP suivants

On stocke le DataFrame brut + la cible binaire pour ne pas avoir à tout refaire au TP 2.


In [ ]:
out_path = "df_raw_with_target.csv"
df_to_save = df.drop(columns=["bm_bin", "Garage_NA"], errors="ignore")
df_to_save.to_csv(out_path, index=False)
print(f"Fichier sauvegardé : {out_path}  (shape={df_to_save.shape})")

---
**Prochain TP : TP 2 — Preprocessing et feature engineering.**
On y traite `VehAge`, `VehMaxSpeed`, on regroupe les `SocioCateg` rares, on encode et on crée un jeu train/test stratifié.
